# Multi files Word Count

In this lab, you will implement a multi-file word count. The texts you will use are freely available books from [Project Gutenberg](http://www.gutenberg.org), including classics such as Lewis Carroll's "Alice in Wonderland" and Jane Austin's "Pride and Prejudice".
We downloaded a few of them and saved them in **wcdata** directory.

## Implementing a Multi-File Word Count

In [14]:
#from pyspark import SparkConf, SparkContext
#sc = SparkContext.getOrCreate(SparkConf())

from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local').appName('WordCountApp').getOrCreate()
sc = spark.sparkContext

## How Spark to do word count operation

Spark uses MapReduce on RDD to do word count.

In [15]:
rdd = sc.parallelize(['hello','world','hello','uic'])
print(type(rdd))
rdd = rdd.map(lambda w: (w,1))
rdd.reduceByKey(lambda x, y: x + y).collect()

<class 'pyspark.rdd.RDD'>


[('hello', 2), ('world', 1), ('uic', 1)]

In [16]:
# we can put them together in one line,as a pyspark.rdd.PipelinedRDD
rdd = sc.parallelize(['hello','world','hello','uic']).map(lambda w: (w,1)).reduceByKey(lambda x, y: x + y)
print(type(rdd))
for w,c in rdd.collect():
    print("%s: %i" % (w, c))

<class 'pyspark.rdd.PipelinedRDD'>
hello: 2
world: 1
uic: 1


## Text file can also be processed as RDD in Spark

In [34]:
#The first line defines a base RDD from an external file.
#This dataset is not loaded in memory or otherwise acted on: lines is merely a pointer to the file.
lines = sc.textFile("data/1.txt")

#The second line defines lineCharacters as the result of a map transformation.
#Again, lineCharacters is not immediately computed, due to laziness.
lineCharacters = lines.map(lambda s: len(s)) #number of characters of each line

print("Characters in each line:")
print(lineCharacters.collect())

#Finally, we run reduce, which is an action.
#At this point Spark breaks the computation into tasks to run on separate machines, and each machine runs
#both its part of the map and a local reduction, returning only its answer to the driver program.

totalCharacters = lineCharacters.reduce(lambda a, b: a + b)
print("Total Characters in the file: %i"% totalCharacters)

Characters in each line:
[579, 767, 300, 618, 755, 910, 696, 579, 768, 299, 618, 755]
Total Characters in the file: 7644


In [22]:
#find the total number of lines of a file
textfile = sc.textFile("data/1.txt")
lines = textfile.count()
print(lines)

12


In [25]:
#Read one text file and do word count operation
#We provide a file name
text_file = sc.textFile('data/1.txt')

#We can also provide a path,such as iitfidf, allowing process all files in that directory
#text_file = sc.textFile('dtat/iitfidf')

#We can aslo use wildcast * to select some files under a directory
#text_file = sc.textFile('data/iitfidf/*.txt')

#MapReduce operation from Spark, 
# the result is sorted in reverse order
wordcounts = text_file.flatMap(lambda line: line.split())\
                    .map(lambda word: (word, 1))\
                    .reduceByKey(lambda a, b: a + b)\
                    .sortBy(lambda x: x[0], True)  # ascending=False

#Option 1: print output
for (word, count) in wordcounts.take(20):
    print("%s: %i" % (word, count))
    
    
# Option 2: save as text file
#wordcounts.saveAsTextFile("output")

"Lorem: 2
"de: 4
'Content: 2
'lorem: 2
(The: 2
(injected: 2
0001: 1
0002: 1
0003: 1
0004: 1
0005: 1
0006: 1
0007: 1
0008: 1
0009: 1
0010: 1
0011: 1
0012: 1
1.10.32: 5
1.10.32.: 1


In [7]:
# find total words in the text file
totalWords = wordcounts.map(lambda x: x[1]).reduce(lambda a, b: a + b)
print("Total Words: %i"% totalWords)   

Total Words: 1229


In [26]:
# find total unique words 
totalUniqueWords = wordcounts.count()
print("Total Words: %i"% totalUniqueWords)   

Total Words: 441


Next, we implement the word-counting program for multiple files.

In [9]:
# actually, the code is almost the same as we did above
rdd_files = sc.textFile("data/wcdata/*.txt")
print(type(rdd_files))

tokens = rdd_files.flatMap(lambda content: content.split())
pairs = tokens.map(lambda word: (word, 1))
wordcounts = pairs.reduceByKey(lambda a, b: a + b)
wc_sorted = wordcounts.sortBy(lambda x: x[1], ascending=False)

#wc_sorted.saveAsTextFile('my_wc')

top10 = wc_sorted.take(10)
for (word, count) in top10:   #wc_sorted.collect():
    print(word, count)

<class 'pyspark.rdd.RDD'>
--> 686829
The 327343
the 238870
of 172294
and 103267
to 76186
a 61426
in 60396
with 50394
Show 44923


In [10]:
# read all files in a directory, do wordcount
import re

rdd_data = sc.textFile('data/iitfidf')
print(type(rdd_data))
lines = rdd_data.count()
print(lines)

wordcount = rdd_data.flatMap(lambda x: [((i.lower()), 1) for i in re.split('\W+', x)]) \
        .reduceByKey(lambda a, b: a + b).sortBy(lambda x: x[1], ascending=False)
wordcount.collect()

<class 'pyspark.rdd.RDD'>
15


[('the', 8),
 ('of', 7),
 ('and', 7),
 ('', 6),
 ('has', 4),
 ('spectre', 3),
 ('into', 3),
 ('new', 3),
 ('is', 3),
 ('my', 3),
 ('a', 3),
 ('society', 3),
 ('love', 2),
 ('old', 2),
 ('antagonisms', 2),
 ('it', 2),
 ('two', 2),
 ('europe', 2),
 ('class', 2),
 ('classes', 2),
 ('epoch', 2),
 ('bourgeoisie', 2),
 ('this', 2),
 ('more', 2),
 ('great', 2),
 ('knitting', 1),
 ('are', 1),
 ('haunting', 1),
 ('all', 1),
 ('have', 1),
 ('holy', 1),
 ('alliance', 1),
 ('police', 1),
 ('spies', 1),
 ('bourgeois', 1),
 ('from', 1),
 ('not', 1),
 ('away', 1),
 ('with', 1),
 ('simplified', 1),
 ('as', 1),
 ('up', 1),
 ('camps', 1),
 ('directly', 1),
 ('facing', 1),
 ('hobby', 1),
 ('passion', 1),
 ('i', 1),
 ('cats', 1),
 ('these', 1),
 ('entered', 1),
 ('to', 1),
 ('exorcise', 1),
 ('pope', 1),
 ('metternich', 1),
 ('guizot', 1),
 ('german', 1),
 ('modern', 1),
 ('that', 1),
 ('sprouted', 1),
 ('ruins', 1),
 ('done', 1),
 ('conditions', 1),
 ('ones', 1),
 ('however', 1),
 ('distinctive', 1),
 ('

In [27]:
# read all files in a directory, do wordcount
# use sparkContext.wholeTextFiles()
# we can find the file name of each text file
import re

rdd_data2 = sc.wholeTextFiles('data/iitfidf')
print(type(rdd_data2))
numFiles = rdd_data2.count()
print(numFiles)

wordcount = rdd_data2.flatMap(lambda x: [((i.lower()), 1) for i in re.split('\W+', x[1])]) \
        .reduceByKey(lambda a, b: a + b).sortBy(lambda x: x[1], ascending=False)
wordcount.collect()

<class 'pyspark.rdd.RDD'>
3


[('the', 8),
 ('and', 7),
 ('of', 7),
 ('has', 4),
 ('is', 3),
 ('my', 3),
 ('a', 3),
 ('spectre', 3),
 ('into', 3),
 ('society', 3),
 ('new', 3),
 ('love', 2),
 ('europe', 2),
 ('old', 2),
 ('this', 2),
 ('', 2),
 ('class', 2),
 ('antagonisms', 2),
 ('it', 2),
 ('classes', 2),
 ('epoch', 2),
 ('bourgeoisie', 2),
 ('more', 2),
 ('two', 2),
 ('great', 2),
 ('knitting', 1),
 ('hobby', 1),
 ('passion', 1),
 ('i', 1),
 ('dogs', 1),
 ('cats', 1),
 ('these', 1),
 ('are', 1),
 ('haunting', 1),
 ('communism', 1),
 ('all', 1),
 ('powers', 1),
 ('have', 1),
 ('entered', 1),
 ('holy', 1),
 ('alliance', 1),
 ('to', 1),
 ('exorcise', 1),
 ('pope', 1),
 ('czar', 1),
 ('metternich', 1),
 ('guizot', 1),
 ('french', 1),
 ('radicals', 1),
 ('german', 1),
 ('police', 1),
 ('spies', 1),
 ('modern', 1),
 ('bourgeois', 1),
 ('that', 1),
 ('sprouted', 1),
 ('from', 1),
 ('ruins', 1),
 ('feudal', 1),
 ('not', 1),
 ('done', 1),
 ('away', 1),
 ('with', 1),
 ('but', 1),
 ('established', 1),
 ('conditions', 1),
 

In [29]:
# read all files in a directory, do wordcount
# use sparkContext.wholeTextFiles()
# we can find the file name of each text file
import os, re

rdd_data2 = sc.wholeTextFiles('data/iitfidf')
print(type(rdd_data2))
numFiles = rdd_data2.count()
print(numFiles)

wordcount = rdd_data2.flatMap(lambda x: [((os.path.basename(x[0]), i.lower()), 1) for i in re.split('\W+', x[1])]) \
            .reduceByKey(lambda a, b: a + b)\
            .sortBy(lambda x: x[1], ascending=False)\
            .map(lambda x:(x[0][1], (x[0][0], x[1])))\
            .reduceByKey(lambda a, b: (a,b))
wordcount.collect()

<class 'pyspark.rdd.RDD'>
3


[('the', (('3.txt', 6), ('2.txt', 2))),
 ('of', (('3.txt', 5), ('2.txt', 2))),
 ('has', ('3.txt', 4)),
 ('my', ('1.txt', 3)),
 ('spectre', ('2.txt', 3)),
 ('and', ((('2.txt', 3), ('1.txt', 2)), ('3.txt', 2))),
 ('society', ('3.txt', 3)),
 ('new', ('3.txt', 3)),
 ('love', ('1.txt', 2)),
 ('a', (('2.txt', 2), ('3.txt', 1))),
 ('europe', ('2.txt', 2)),
 ('class', ('3.txt', 2)),
 ('antagonisms', ('3.txt', 2)),
 ('it', ('3.txt', 2)),
 ('classes', ('3.txt', 2)),
 ('epoch', ('3.txt', 2)),
 ('bourgeoisie', ('3.txt', 2)),
 ('more', ('3.txt', 2)),
 ('into', (('3.txt', 2), ('2.txt', 1))),
 ('two', ('3.txt', 2)),
 ('great', ('3.txt', 2)),
 ('knitting', ('1.txt', 1)),
 ('is', ((('1.txt', 1), ('2.txt', 1)), ('3.txt', 1))),
 ('hobby', ('1.txt', 1)),
 ('passion', ('1.txt', 1)),
 ('i', ('1.txt', 1)),
 ('dogs', ('1.txt', 1)),
 ('cats', ('1.txt', 1)),
 ('these', ('1.txt', 1)),
 ('are', ('1.txt', 1)),
 ('haunting', ('2.txt', 1)),
 ('communism', ('2.txt', 1)),
 ('all', ('2.txt', 1)),
 ('powers', ('2.txt', 

##  You can also provide HDFS URI for batch processing

Assuming you have HDFS server @ johnzhang10:9000

Here **johnzhang10** is your hadoop namenode machine name. In order to run it, you have to start hadoop first.(run start-dfs.sh and start-yarn.sh)

In [ ]:
from pyspark.sql import SparkSession
# suppose johnzhang10 is the name of your master node
spark = SparkSession.builder.master('spark://johnzhang10:7077').appName('MyWordCount').getOrCreate()
sc = spark.sparkContext

# in order to read data from hdfs format, you have to start hadoop
# start-dfs.sh && start-yarn.sh
# make sure you have some txt files in /input directory
# suppose johnzhang10 is your master name.
# text_file = sc.textFile('hdfs://johnzhang10:9000/input/*.txt')

# read data from local file system of each worker node
# txt files should exist in each worker node
# split data files among worker nodes

text_file = sc.textFile('/root/*.txt')
wordcounts = text_file.flatMap(lambda line: line.split(" ")).map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)
wordcounts.saveAsTextFile('myoutput')
print(wordcounts.collect())


You can also find out the results in each work node
The wordcounts are saved distributedly in **myoutput** directory in each work node

## You can also run this command from CLI of your Spark Master node:
#### spark-submit --master spark://johnzhang10:7077 /root/spark/examples/src/main/python/wordcount.py "hdfs://johnzhang10:9000/input/*.txt"

In [ ]:
sc.close()